# Task 2
1、使用逻辑回归、SVM支持向量机对手写数字案例进行训练和预测

* 观察运行时间
* 观察准确率



2、对手写数字数据进行PCA降维，然后使用逻辑回归、SVM支持向量机对降维后的数据进行训练和预测

* 观察运行时间
* 观察准确率

## 导入包

In [9]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from functools import partial
from typing import Callable, Iterable, Optional, Sequence, TypedDict,Union,List,Dict,Literal
from enum import Enum,StrEnum,unique

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from dotenv import load_dotenv
from loguru import logger
from pathlib import Path
from joblib import Parallel,delayed,Memory,dump,load

from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline,Parallel
from sklearn.svm import SVC

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

load_dotenv("./.env")

sns.set_theme(style="whitegrid")

# Choose available CJK fonts first to avoid findfont / missing glyph warnings.
preferred_cjk_fonts = [
    "Noto Sans CJK SC",
    "Noto Sans CJK TC",
    "Noto Sans CJK JP",
    "Microsoft YaHei",
    "SimHei",
    "Arial Unicode MS",
]
installed_fonts = {font.name for font in fm.fontManager.ttflist}
available_fonts = [name for name in preferred_cjk_fonts if name in installed_fonts]
plt.rcParams["font.sans-serif"] = available_fonts + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [2]:
@dataclass
class Config:
    data_path:Optional[Path]=Path('./data/inputs/digits.csv')
    openai_apikey:Optional[str]=os.getenv("OPENAI_API_KEY")
    openai_base_url:Optional[str]=os.getenv("OPENAI_BASE_URL")
    openai_model:Optional[str]=os.getenv("OPENAI_MODEL")
    
cfg=Config

## 载入数据

In [3]:
class Data:
    """A class to handle data preprocessing and splitting.
    
    This class encapsulates the data loading, feature separation, and 
    train-test split functionality.
    """
    
    def __init__(self,file_path:Optional[Path|str]) -> None:
        """Initialize the Data object with raw data.
        
        Args:
            raw_data: A pandas DataFrame containing the raw data with a 'label' column.
        """
        self._data = self._load_data(file_path)
        self._X = self._data.drop(columns=["label"])
        self._y = self._data["label"]
    
    @property
    def X(self) -> pd.DataFrame:
        """Get the feature data."""
        return self._X
    
    @property
    def y(self) -> pd.Series:
        """Get the target labels."""
        return self._y
    
    def split_data(self, test_size: float = 0.2, random_state: int = 42) -> None:
        """Split the data into training and testing sets.
        
        Args:
            test_size: Proportion of dataset to include in the test split.
            random_state: Controls the shuffling applied to the data before applying the split.
        """
        # Split the data using stratified sampling to maintain label distribution
        (
            self._X_train,
            self._X_test,
            self._y_train,
            self._y_test
        ) = train_test_split(
            self._X,
            self._y,
            test_size=test_size,
            random_state=random_state,
            stratify=self._y
        )
    
    @property
    def X_train(self) -> pd.DataFrame:
        """Get the training features."""
        return self._X_train
    
    @property
    def X_test(self) -> pd.DataFrame:
        """Get the test features."""
        return self._X_test
    
    @property
    def y_train(self) -> pd.Series:
        """Get the training labels."""
        return self._y_train
    
    @property
    def y_test(self) -> pd.Series:
        """Get the test labels."""
        return self._y_test
    def get_info(self)-> Optional[pd.DataFrame]:
        """获取数据信息

        Returns:
            object: pd
        """
        return self._data.info()
    def get_describe(self)-> Optional[pd.DataFrame]:
        """获取数据描述性分析

        Returns:
            object: pd
        """
        return self._data.describe()
    def get_head(self,n:Optional[int])-> Optional[pd.DataFrame]:
        """获取数据前几行信息

        Returns:
            object: pd
        """
        if n is None:
            return self._data.head()
        else:
            return self._data.head(n)
    def _load_data(self,data_path: Optional[Union[Path, str]]) -> pd.DataFrame:
        """
        从CSV文件加载数据

        Args:
            data_path: CSV文件路径，可以是Path对象或字符串
            
        Returns:
            pandas.DataFrame: 加载的数据框
            
        Raises:
            FileNotFoundError: 当文件不存在时
            pd.errors.EmptyDataError: 当文件为空时
            pd.errors.ParserError: 当CSV解析失败时
            UnicodeDecodeError: 当编码错误时
            ValueError: 当输入参数无效时
        """
        # 输入验证
        if data_path is None:
            raise ValueError("data_path cannot be None")
        
        # 转换为Path对象以便统一处理
        path = Path(data_path)
        
        # 检查文件是否存在
        if not path.exists():
            raise FileNotFoundError(f"Data file not found: {path}")
        
        # 检查是否为文件（而不是目录）
        if not path.is_file():
            raise ValueError(f"Path is not a file: {path}")
        
        try:
            # 尝试使用UTF-8编码加载文件
            df = pd.read_csv(path, encoding='utf-8')
            logger.info(f"Successfully loaded data from {path}")
            return df
            
        except UnicodeDecodeError:
            # 如果UTF-8失败，尝试其他常见编码
            logger.warning(f"Failed to decode {path} with UTF-8, trying other encodings...")
            try:
                df = pd.read_csv(path, encoding='latin-1')
                logger.info(f"Loaded data from {path} using latin-1 encoding")
                return df
            except UnicodeDecodeError:
                # 最后尝试系统默认编码
                logger.warning(f"Failed to decode {path} with latin-1, trying system default...")
                df = pd.read_csv(path)
                logger.info(f"Loaded data from {path} using system default encoding")
                return df
                
        except Exception as e:
            # 记录详细错误信息
            logger.error(f"Error loading data from {path}: {str(e)}")
            raise
data=Data(cfg.data_path)
data.split_data()

2026-04-23 20:21:59.133 | INFO     | __main__:_load_data:127 - Successfully loaded data from data/inputs/digits.csv


## 建模

### 非降维

In [4]:
class LRSolver(StrEnum):
    NEWTON_CG ='newton-cg'
    LBFGS = 'lbfgs'
    SAGA = 'saga'
    SAG='sag'

In [12]:
def logistic_regression(
    data:Data,
    Cs: Sequence[float],
    cv: Optional[int],
    solvers: Sequence[str],
    random_state:int=42,
    max_iter:int=5000
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Perform logistic regression with hyperparameter tuning using cross-validation.
    
    Args:
        Cs: Regularization strength values to try
        cv: Number of cross-validation folds
        solvers: List of solvers to try
        
    Returns:
        Dictionary containing the best parameters and performance metrics,
        or None if no valid configuration was found
    """
    # Initialize best score and parameters
    best_score = float('-inf')
    best_params = {
        "solver": None,
        "score": best_score,
        "C": None,
        "coef": None,
        "intercept": None
    }
    
    # Validate inputs
    if not Cs or not solvers:
        return None
    
    # Use a more robust approach for handling potential errors
    for solver in solvers:
        try:
            # Create logistic regression model with cross-validation
            lr = LogisticRegressionCV(
                Cs=Cs, 
                cv=cv, 
                solver=solver,
                random_state=42,  
                max_iter=max_iter
            )
            
            # Fit the model
            lr.fit(data.X_train, data.y_train)
            
            # Calculate test score
            current_score = lr.score(data.X_test, data.y_test)
            
            # Update best parameters if current is better
            if current_score > best_score:
                best_score = current_score
                best_params["C"] = lr.C_[0] if hasattr(lr.C_, '__len__') else lr.C_
                best_params["coef"] = lr.coef_.copy() if lr.coef_ is not None else None
                best_params["intercept"] = lr.intercept_[0] if hasattr(lr.intercept_, '__len__') else lr.intercept_
                best_params["score"] = best_score
                best_params["solver"] = solver
                
        except Exception as e:
            # Log warning instead of crashing
            logger.exception(f"Failed to evaluate solver {solver}: {str(e)}")
            continue
    
    # Return best parameters or None if no valid solution found
    return best_params if best_score != float('-inf') else None

In [10]:
def randomCV_svm(
    data: Data,
    Cs: Sequence[float],
    kernel: str | Sequence[str] = "rbf",
    gammas: Sequence[float] | str = "auto",
    cv: Optional[int] = 5,
    random_state: int = 42
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    使用随机搜索进行SVM超参数调优
    
    Args:
        data: 包含训练数据的Data对象
        Cs: C参数候选值序列
        kernel: 核函数类型或候选值序列
        gammas: gamma参数候选值序列或"auto"
        cv: 交叉验证折数
        random_state: 随机种子
    
    Returns:
        最佳参数字典，包含C、kernel、gamma、score和support_vectors
    """
    
    # 输入验证
    if not Cs or not isinstance(Cs, (list, tuple)):
        raise ValueError("Cs must be a non-empty sequence")
    
    if cv is not None and cv <= 0:
        raise ValueError("cv must be positive")
    
    # 参数预处理
    kernel_values = [kernel] if isinstance(kernel, str) else list(kernel)
    gamma_values = [gammas] if isinstance(gammas, str) else list(gammas)
    c_values = list(Cs)
    
    # 避免空参数组合
    if not c_values or not kernel_values or not gamma_values:
        return None
    
    # 计算可能的组合数量
    total_combinations = len(c_values) * len(kernel_values) * len(gamma_values)
    n_iter = min(30, total_combinations)
    
    # 构建参数分布
    param_distributions = {
        "C": c_values,
        "kernel": kernel_values,
        "gamma": gamma_values
    }
    
    # 创建随机搜索模型
    model = RandomizedSearchCV(
        estimator=SVC(random_state=random_state),
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        random_state=random_state,
        n_jobs=-1,  # 利用所有CPU核心
        verbose=0,
        scoring='accuracy'  # 明确指定评分标准
    )
    
    # 执行拟合
    try:
        model.fit(data.X_train, data.y_train)
        
        # 检查是否找到了有效的解决方案
        if hasattr(model, 'best_score_') and model.best_score_ != float('-inf'):
            # 提取最佳参数
            best_params = model.best_params_.copy()
            best_params["score"] = model.best_score_
            best_params["support_vectors"] = model.best_estimator_.support_vectors_
            
            return best_params
            
    except Exception as e:
        # 错误处理：记录错误并返回None
        logger.exception(f"模型训练失败: {str(e)}")
        return None
    
    return None

In [11]:
def GridCV_svm(
    data: Data,
    Cs: Sequence[float],
    kernel: str | Sequence[str] = "rbf",
    gammas: Sequence[float] | str = "auto",
    cv: Optional[int] = 5,
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Improved version with better error handling and parameter validation.
    
    Args:
        data: Training data containing X_train and y_train
        Cs: List of C values to try (must be positive)
        kernel: Kernel type(s) to try (should be in ['linear', 'poly', 'rbf', 'sigmoid'])
        gammas: Gamma values to try or 'auto' string
        cv: Number of cross-validation folds (positive integer)
    
    Returns:
        Dictionary with best parameters and results, or None if no valid result
    """
    # Input validation
    if not Cs:
        raise ValueError("Cs sequence cannot be empty")
    
    if cv is not None and cv <= 0:
        raise ValueError("cv must be a positive integer")
    
    # Validate that C values are positive
    if any(c <= 0 for c in Cs):
        raise ValueError("All C values must be positive")
    
    # Initialize best score
    best_score = float("-inf")
    
    # Handle kernel parameter
    kernel_values = [kernel] if isinstance(kernel, str) else list(kernel)
    
    # Handle gamma parameter
    gamma_values = [gammas] if isinstance(gammas, str) else list(gammas)
    
    # Convert Cs to list
    c_values = list(Cs)
    
    # Create parameter grid
    param_grid = {
        "C": c_values,
        "kernel": kernel_values,
        "gamma": gamma_values
    }
    
    # Create grid search model with improved settings
    model = GridSearchCV(
        SVC(random_state=42),  # Set random state for reproducibility
        param_grid=param_grid,
        cv=cv,
        n_jobs=-1,  # Parallel processing for speed
        verbose=0,
        scoring='accuracy'  # Explicitly specify scoring metric
    )
    
    try:
        # Fit the model
        model.fit(data.X_train, data.y_train)
        
        # Check if we found a valid solution
        if model.best_score_ == float("-inf"):
            return None
            
        # Extract results
        best_params = model.best_params_.copy()
        best_score = model.best_score_
        
        # Add additional information
        best_params["score"] = best_score
        best_params["support_vectors"] = model.best_estimator_.support_vectors_
        
        return best_params
        
    except Exception as e:
        # Log error but don't crash the application
        logger.exception(f"Error during grid search: {e}")
        return None

### PCA降维之后建模

In [14]:
def pca_logistic_regression(
    data: Data,
    Cs: Sequence[float],
    cv: Optional[int],
    solvers: Sequence[str],
    random_state: int = 42,
    max_iter: int = 5000,
    n_components: Union[float, int, str] = 0.95
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Perform logistic regression with hyperparameter tuning using cross-validation.

    Args:
        data: An object containing X_train, y_train, X_test, y_test attributes.
        Cs: Regularization strength values to try.
        cv: Number of cross-validation folds; if None, defaults to 5.
        solvers: List of solvers to try.
        random_state: Random state used for reproducibility.
        max_iter: Maximum number of iterations for the solver.
        n_components: Number of components to keep in PCA (can be float, int, or 'mle').

    Returns:
        Dictionary containing the best parameters and performance metrics,
        or None if no valid configuration was found.
    """
    # 初始化最佳得分及参数
    best_score = float('-inf')
    best_params = {
        "solver": None,
        "score": best_score,
        "C": None,
        "coef": None,
        "intercept": None
    }

    # 输入验证
    if not Cs or not solvers:
        logger.warning("No regularization strengths or solvers provided.")
        return None

    # 尝试不同 solver
    for solver in solvers:
        try:
            # 构建 pipeline：PCA + LogisticRegressionCV
            model = Pipeline([
                ('pca', PCA(n_components=n_components, random_state=random_state)),
                ('log_reg', LogisticRegressionCV(
                    Cs=Cs,
                    cv=cv or 5,  # 默认为5折交叉验证
                    solver=solver,
                    random_state=random_state,
                    max_iter=max_iter
                ))
            ])

            # 训练模型
            model.fit(data.X_train, data.y_train)

            # 获取最优模型
            log_reg = model.named_steps['log_reg']

            # 测试集评分
            current_score = model.score(data.X_test, data.y_test)

            # 更新最佳参数
            if current_score > best_score:
                best_score = current_score
                best_params.update({
                    "C": log_reg.C_[0] if hasattr(log_reg.C_, '__len__') else log_reg.C_,
                    "coef": log_reg.coef_.copy() if log_reg.coef_ is not None else None,
                    "intercept": (
                        log_reg.intercept_[0]
                        if hasattr(log_reg.intercept_, '__len__')
                        else log_reg.intercept_
                    ),
                    "score": best_score,
                    "solver": solver
                })

        except Exception as e:
            logger.exception(f"Failed to evaluate solver '{solver}': {str(e)}")
            continue

    # 如果没有找到有效配置，则返回 None
    return best_params if best_score != float('-inf') else None

In [15]:
def pca_gridcv_svm(
    data,
    cs: Sequence[float],
    kernel: Union[str, Sequence[str]] = "rbf",
    gammas: Sequence[float] | str = "auto",
    cv: Optional[int] = 5,
    n_components: Union[float, int, str] = 0.95
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Performs PCA + SVM Grid Search with cross-validation.

    Args:
        data: Training data object with attributes X_train and y_train.
        cs: List of C values to test; all must be positive.
        kernel: Kernel type(s) to try (e.g., 'linear', 'rbf', etc.)
        gammas: Gamma values to try or 'auto'.
        cv: Number of cross-validation folds (default=5).
        n_components: Number of components for PCA (can be int, float, or 'mle').

    Returns:
        Dictionary containing best parameters and evaluation metrics,
        or None if an error occurred during fitting.
    """
    # --- 输入验证 ---
    if not cs:
        raise ValueError("Parameter 'cs' must contain at least one value.")

    if cv is not None and cv <= 0:
        raise ValueError("Parameter 'cv' must be a positive integer.")

    if any(c <= 0 for c in cs):
        raise ValueError("All values in 'cs' must be positive.")

    # Validate data structure
    try:
        X_train, y_train = check_X_y(data.X_train, data.y_train)
    except Exception as e:
        logger.error(f"Invalid training data provided: {e}")
        return None

    # --- 初始化参数网格 ---
    kernel_list = [kernel] if isinstance(kernel, str) else list(kernel)
    gamma_list = [gammas] if isinstance(gammas, str) else list(gammas)
    c_list = list(cs)

    param_grid = {
        "GridCV_svm__C": c_list,
        "GridCV_svm__kernel": kernel_list,
        "GridCV_svm__gamma": gamma_list
    }

    # --- 构建 Pipeline ---
    pipeline = Pipeline([
        ('pca', PCA(n_components=n_components)),
        ('GridCV_svm', GridSearchCV(
            estimator=SVC(random_state=42),
            param_grid=param_grid,
            cv=cv,
            n_jobs=-1,
            verbose=0,
            scoring='accuracy'
        ))
    ])

    # --- 执行拟合并获取结果 ---
    try:
        pipeline.fit(X_train, y_train)

        # 如果没有找到有效的模型（例如因为所有组合都失败）
        if pipeline.named_steps['GridCV_svm'].best_score_ == float('-inf'):
            logger.warning("No valid model found during grid search.")
            return None

        # 提取最佳参数及支持向量
        best_params = pipeline.named_steps['GridCV_svm'].best_params_.copy()
        best_score = pipeline.named_steps['GridCV_svm'].best_score_

        # 添加额外信息
        best_params["score"] = best_score
        best_params["support_vectors"] = pipeline.named_steps['GridCV_svm'].best_estimator_.support_vectors_

        return best_params

    except Exception as e:
        logger.exception(f"An error occurred during grid search: {e}")
        return None